# 04 · Assignment: model comparison report

1. Write **5 test questions** on one topic you know well
2. Run each on **2 models** (Gemini and Groq) at temperature 0.2
3. Save the answer, tokens in and out, and latency to `results.csv`
4. Rate each answer in the CSV: **correct**, **partly correct** or **wrong**
5. In the README: which model would you choose, and why?

**Topic chosen:** Python programming fundamentals — every question has one objectively
correct answer, so the ratings are defensible.

**Submit** your GitHub repo link on Google Classroom.

In [1]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads the .env file in this folder

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

# Never print the key itself, only whether it was found
print("Gemini key found:", bool(os.getenv("GEMINI_API_KEY")))
print("Groq key found:  ", bool(os.getenv("GROQ_API_KEY")))

Gemini key found: True
Groq key found:   True


## Step 1 · Test questions + grading notes

`GOLD` is **not** sent to the models. It is your answer key for rating the
answers after the run — it keeps the correct / partly correct / wrong calls
objective instead of a gut feeling.

In [2]:
TOPIC = "Python programming fundamentals"

QUESTIONS = [
    "In Python, what is the difference between a list and a tuple? Answer in 2-3 sentences.",
    "What does the `self` parameter in a Python class method refer to? One or two sentences.",
    "Write a one-line Python expression that returns a list of the squares of the even numbers from 0 to 9. Show only the code.",
    "What is the output of: print(type([]) is list)? Answer with the output only.",
    "Explain the difference between `==` and `is` in Python in 2-3 sentences.",
]

# Answer key — used only for rating, never sent to the model
GOLD = {
    0: "Lists are mutable (can be changed after creation); tuples are immutable. "
       "Lists use [], tuples use ().",
    1: "self refers to the instance of the class on which the method was called.",
    2: "[x**2 for x in range(10) if x % 2 == 0]  ->  [0, 4, 16, 36, 64]",
    3: "True",
    4: "== compares values (equality); is compares identity (same object in memory).",
}

for i, q in enumerate(QUESTIONS, 1):
    print(f"Q{i}: {q}")

Q1: In Python, what is the difference between a list and a tuple? Answer in 2-3 sentences.
Q2: What does the `self` parameter in a Python class method refer to? One or two sentences.
Q3: Write a one-line Python expression that returns a list of the squares of the even numbers from 0 to 9. Show only the code.
Q4: What is the output of: print(type([]) is list)? Answer with the output only.
Q5: Explain the difference between `==` and `is` in Python in 2-3 sentences.


## Step 2 · The models to compare

In [3]:
MODELS = {
    "gemini": (OpenAI(api_key=os.getenv("GEMINI_API_KEY"), base_url=GEMINI_BASE_URL, timeout=30), GEMINI_MODEL),
    "groq": (OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url=GROQ_BASE_URL, timeout=30), GROQ_MODEL),
}

for provider, (client, model) in MODELS.items():
    print(provider, "->", model)

gemini -> gemini-3.8-flash
groq -> openai/gpt-oss-20b


## Step 3 · Run every question on every model

Includes the retry pattern from notebook 02 (429 / timeout / connection errors,
back-off 2s, 4s, 8s). One model failing does not kill the whole run — the error
is recorded in the row so the CSV still tells the full story.

In [4]:
from openai import RateLimitError, APITimeoutError, APIConnectionError

RETRYABLE = (RateLimitError, APITimeoutError, APIConnectionError)


def ask(client, model, question, retries=3):
    messages = [{"role": "user", "content": question}]
    for attempt in range(1, retries + 1):
        try:
            return client.chat.completions.create(
                model=model, messages=messages, temperature=0.2
            )
        except RETRYABLE as e:
            wait = 2 ** attempt
            print(f"  retry {attempt} in {wait}s: {type(e).__name__}")
            time.sleep(wait)
    raise RuntimeError("all retries failed")


results = []

for provider, (client, model) in MODELS.items():
    for qi, q in enumerate(QUESTIONS):
        row = {"provider": provider, "model": model, "question": q,
               "tokens_in": None, "tokens_out": None, "latency_s": None,
               "answer": "", "error": "", "rating": ""}
        try:
            start = time.perf_counter()
            r = ask(client, model, q)
            row["latency_s"] = round(time.perf_counter() - start, 2)
            row["answer"] = r.choices[0].message.content.strip()
            if r.usage:
                row["tokens_in"] = r.usage.prompt_tokens
                row["tokens_out"] = r.usage.completion_tokens
            print(f"{provider} Q{qi+1}: done ({row['latency_s']}s)")
        except Exception as e:
            row["error"] = type(e).__name__
            print(f"{provider} Q{qi+1}: FAILED ({row['error']})")
        results.append(row)

print(f"\nCollected {len(results)} rows")

gemini Q1: done (8.41s)
gemini Q2: done (11.41s)
gemini Q3: done (21.63s)
  retry 1 in 2s: RateLimitError
  retry 2 in 4s: RateLimitError
  retry 3 in 8s: RateLimitError
gemini Q4: FAILED (RuntimeError)
gemini Q5: done (9.37s)
groq Q1: done (0.65s)
groq Q2: done (0.46s)
groq Q3: done (0.61s)
groq Q4: done (0.47s)
groq Q5: done (0.48s)

Collected 10 rows


## Step 4 · Save to results.csv

In [5]:
import csv

with open("results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

print("Saved", len(results), "rows to results.csv")

Saved 10 rows to results.csv


## Step 5 · Rate the answers

Open `results.csv` and fill in the **rating** column: `correct`, `partly correct`
or `wrong`. Compare each answer against the `GOLD` key from Step 1.

Rating guide:

- **correct** — all key facts right, no misleading claims
- **partly correct** — main idea right but a fact missing or slightly off
- **wrong** — incorrect, off topic, or an error row

After saving the CSV with ratings, run the next cell. It reloads the file and
prints the summary plus a ready-to-paste paragraph for your README.

In [1]:
import csv

with open("results.csv", newline="", encoding="utf-8") as f:
    rated = list(csv.DictReader(f))

VALID = {"correct", "partly correct", "wrong"}
unrated = [r for r in rated if r["rating"] not in VALID]
if unrated:
    print(f"WARNING: {len(unrated)} row(s) still need a rating (correct / partly correct / wrong)")

print(f"{'provider':<10} {'correct':>8} {'partly':>7} {'wrong':>6} {'errors':>7} {'avg lat':>8} {'avg tok out':>12}")
summary = {}
for provider in sorted({r["provider"] for r in rated}):
    rows = [r for r in rated if r["provider"] == provider]
    scored = [r for r in rows if r["rating"] in VALID]
    lat = [float(r["latency_s"]) for r in rows if r["latency_s"]]
    tok = [int(r["tokens_out"]) for r in rows if r["tokens_out"]]
    n = lambda s: sum(1 for r in scored if r["rating"] == s)
    summary[provider] = {
        "model": rows[0]["model"],
        "correct": n("correct"), "partly": n("partly correct"), "wrong": n("wrong"),
        "errors": sum(1 for r in rows if r["error"]),
        "avg_latency": round(sum(lat) / len(lat), 2) if lat else None,
        "avg_tokens_out": round(sum(tok) / len(tok)) if tok else None,
    }
    s = summary[provider]
    print(f"{provider:<10} {s['correct']:>8} {s['partly']:>7} {s['wrong']:>6} {s['errors']:>7} {str(s['avg_latency']):>8} {str(s['avg_tokens_out']):>12}")

print("\n--- Ready-to-paste README paragraph ---")
for provider, s in summary.items():
    print(f"- **{provider}** ({s['model']}): {s['correct']} correct, {s['partly']} partly correct, "
          f"{s['wrong']} wrong, {s['errors']} errors, avg latency {s['avg_latency']}s, avg {s['avg_tokens_out']} tokens out")

provider    correct  partly  wrong  errors  avg lat  avg tok out
gemini            4       0      1       1    12.71           56
groq              5       0      0       0     0.53          128

--- Ready-to-paste README paragraph ---
- **gemini** (gemini-3.8-flash): 4 correct, 0 partly correct, 1 wrong, 1 errors, avg latency 12.71s, avg 56 tokens out
- **groq** (openai/gpt-oss-20b): 5 correct, 0 partly correct, 0 wrong, 0 errors, avg latency 0.53s, avg 128 tokens out


## Submission checklist

1. `results.csv` has all 10 rows rated (5 questions × 2 models)
2. README's *Assignment 2* section has the pasted summary + your model choice
3. `git status` shows no `.env` — then commit and push
4. Submit the GitHub repo link on Google Classroom